# 10 Tests for Code, Data, and Model Logic

After refactoring, we can test important project behavior without running full notebooks. Tests are a safety net: they make sure that future changes do not silently break cleaning rules, feature definitions, evaluation assumptions, or prediction behavior.

## 1. What We Test First

The first test suite is intentionally small. It focuses on behavior that is easy to break and important for the project:

- data cleaning rules for `TotalCharges` and `ChurnBinary`
- deterministic feature engineering such as `TenureGroup` and add-on service counts
- probability-based business KPI logic
- prediction threshold behavior
- availability of the command-line pipeline interface

We distinguish two test types:

- **Unit tests** check one function or one small piece of logic in isolation. They should be fast, focused, and easy to understand.
- **Integration tests** check whether multiple components work together. They are often a bit slower or broader. Our first integration test checks that the pipeline CLI exposes the expected commands.

These are not yet exhaustive production tests. They are the first practical layer of automated checks.

## 2. Test Files and Fixtures

The tests live in `tests/`:

```text
tests/
  conftest.py
  test_dataset.py
  test_features.py
  test_evaluation.py
  test_predict.py
  test_pipeline_cli.py
```

`conftest.py` contains shared fixtures. A fixture is reusable test setup. In this project, `minimal_telco_input` provides a tiny Telco-style input row that can be reused across tests.

The tests use small artificial data examples instead of the full Telco dataset where possible. This keeps the tests fast and makes the expected behavior easy to inspect.

## 3. Test Naming Convention

The test function names are intentionally descriptive. With `pytest`, long names are common because the test name appears directly in the terminal output when a test fails. A good test name should make the expected behavior visible without opening the test file first.

We use this simple pattern:

```text
test_<function_or_component>_<expected_behavior>
```

Example:

```python
def test_create_business_features_adds_expected_columns():
    ...
```

Read as a sentence, this means: `create_business_features` should add the expected columns. If this test fails, the terminal already tells us which behavior changed.

Short names such as `test_features()` are easier to type, but they are less helpful when debugging. The goal is not to make names as short as possible. The goal is to make the tested behavior explicit.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "tests").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sorted(path.name for path in (PROJECT_ROOT / "tests").glob("test_*.py"))

['test_dataset.py',
 'test_evaluation.py',
 'test_features.py',
 'test_pipeline_cli.py',
 'test_predict.py']

## 4. Run the Tests

From the repository root, all tests can be run with:

```bash
python -m pytest
```

Only unit tests:

```bash
python -m pytest -m unit
```

Only integration tests:

```bash
python -m pytest -m integration
```

The same commands can later be used in GitHub Actions.

In [2]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Tests failed")

============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-9.1.0, pluggy-1.6.0
rootdir: /home/cbaldermann/Projekte/ADS_II_MLOPS/2026-06-ads-II-master
configfile: pyproject.toml
testpaths: tests
plugins: anyio-4.13.0
collected 7 items

tests/test_dataset.py ..                                                 [ 28%]
tests/test_evaluation.py .                                               [ 42%]
tests/test_features.py ..                                                [ 71%]
tests/test_pipeline_cli.py .                                             [ 85%]
tests/test_predict.py .                                                  [100%]

============================== 7 passed in 1.79s ===============================



## 4. What These Tests Do Not Cover Yet

This first suite does not fully test model quality, drift, deployment, or every possible malformed input. That would be too much for the first testing step.

Useful next additions are:

- data schema tests for required columns and allowed values
- model behavior tests, for example minimum expected F1 or ROC-AUC on a fixed test set
- pipeline smoke tests with a small fixture dataset
- API contract tests once model serving is introduced
- CI checks that run tests automatically on pull requests

The key idea is incremental: start with fast unit tests for stable assumptions, then add integration tests where project risk increases.